# Forested versus open pixels

The analyses keep pixels with forest cover fraction (FCF) ≤ 50 %. This notebook compares them with the pixels the
filter removes, FCF > 50 % ("forested"), in the same range, elevation band and aspect bin, from the forest-cover key
of the sparse table: the 11-year median onset and MAD, the lapse rate and the sunny–shady difference of each
population, and the year-to-year anomalies (how closely forested anomalies track open ones, and their spring-temperature
slopes).

```
reads                                                      writes (results/<version>/)
─────                                                      ──────────────────────────
data/aggregation/<v>/sparse_mountain_ranges.parquet ──┐     forest_vs_open_by_range.csv   one row per range
results/<v>/mountain_range_statistics.csv ────────────┴──►  forest_vs_open_summary.json
```

A bin enters a matched comparison only if both populations have at least `MIN_PIXELS` pixels in it; the per-range
difference is weighted by the smaller of the two counts. Pixels with no FCF value (north of the FCF product) are in
neither population.

In [ ]:
import json

import numpy as np
import pandas as pd
import scipy.stats as sps
import xarray as xr

from gsro_analysis import aggregate, paths, range_metrics as rm, results, settings

In [ ]:
config = settings.load_config()
VERSION = config.version
WATER_YEARS = [int(y) for y in config.water_years]
UNIT = 'mountain_ranges'
aggregation_dir = paths.aggregation_dir(UNIT, VERSION)
results_dir = paths.resultsdir(UNIT, VERSION)

OPEN_CLASSES = aggregate.FILTER_VIEWS['fcf_lte_50']     # FCF 0-50 %
FOREST_CLASSES = list(range(5, 10))                     # FCF (50, 100] %
MIN_PIXELS = rm.MIN_PIXELS_PER_BIN
MIN_FOREST_PIXELS = 5000          # a range is compared only with this many forested pixels
MIN_MATCHED_BINS = 5
MIN_MATCHED_YEARS = 8             # interannual comparison: years with >= MIN_MATCHED_BINS matched bins

In [ ]:
sparse_path = aggregation_dir / f'sparse_{UNIT}.parquet'
keys = ('elevation', 'aspect')
open_ds = aggregate.view(aggregate.load_sparse(sparse_path, keys, fcf=OPEN_CLASSES), UNIT, keys, WATER_YEARS, fcf=OPEN_CLASSES)
forest_ds = aggregate.view(aggregate.load_sparse(sparse_path, keys, fcf=FOREST_CLASSES), UNIT, keys, WATER_YEARS, fcf=FOREST_CLASSES)
open_ds, forest_ds = xr.align(open_ds, forest_ds, join='inner')
forest_ds

In [ ]:
statistics_df = pd.read_csv(results_dir / 'mountain_range_statistics.csv').set_index('name')
id_to_name = dict(zip(statistics_df['GMBA_V2_ID'].astype(int), statistics_df.index))
names = [id_to_name.get(int(i)) for i in open_ds['mountain_range'].values]
open_ds = open_ds.assign_coords(mountain_range=names).sel(mountain_range=[n is not None for n in names])
forest_ds = forest_ds.assign_coords(mountain_range=names).sel(mountain_range=[n is not None for n in names])
print(f"{open_ds.sizes['mountain_range']} ranges in both views")

## 1. Climatology: matched-bin onset and MAD differences

In [ ]:
open_v, forest_v = rm.base_view(open_ds, MIN_PIXELS), rm.base_view(forest_ds, MIN_PIXELS)
onset_diff, matched_bins = rm.matched_difference(forest_v, open_v, 'runoff_onset_median', MIN_PIXELS)
mad_diff, _ = rm.matched_difference(forest_v, open_v, 'runoff_onset_mad', MIN_PIXELS)
forest_px = forest_ds['runoff_onset_median_n'].fillna(0).sum(rm.BIN_DIMS).to_pandas()
open_px = open_ds['runoff_onset_median_n'].fillna(0).sum(rm.BIN_DIMS).to_pandas()
by_range_df = pd.DataFrame({'forest_pixels': forest_px, 'open_pixels': open_px,
                            'forest_share': forest_px / (forest_px + open_px), 'matched_bins': matched_bins,
                            'onset_forest_minus_open_d': onset_diff, 'mad_forest_minus_open_d': mad_diff})
by_range_df.index.name = 'name'
compared = (by_range_df['matched_bins'] >= MIN_MATCHED_BINS) & (by_range_df['forest_pixels'] >= MIN_FOREST_PIXELS)
by_range_df['compared'] = compared
by_range_df[compared].describe().round(2)

## 2. Topography of each population: lapse rate and sunny–shady difference

In [ ]:
elevation = open_v['elevation'].values
for label, v in (('open', open_v), ('forest', forest_v)):
    by_range_df[f'lapse_rate_{label}'] = [rm.lapse_fit(v['runoff_onset_median'].sel(mountain_range=r),
                                                       v['runoff_onset_median_n'].sel(mountain_range=r), elevation)
                                          for r in v['mountain_range'].values]
    by_range_df[f'sunny_shady_{label}_d'] = rm.sunny_shady(v)[0]['sunny_shady_d'].values
by_range_df.loc[compared, ['lapse_rate_open', 'lapse_rate_forest', 'sunny_shady_open_d', 'sunny_shady_forest_d']].describe().round(2)

## 3. Year to year: do forested anomalies track open ones?

Per range-year, the pixel-weighted mean anomaly of each population over the bins matched that year (both ≥
`MIN_PIXELS` pixels that year); then per range their correlation, s.d. and spring-temperature slopes.

In [ ]:
n_f, n_o = forest_ds['runoff_onset_n'], open_ds['runoff_onset_n']
ok = (n_f >= MIN_PIXELS) & (n_o >= MIN_PIXELS)
w = np.minimum(n_f, n_o).where(ok)
forest_anom = (forest_ds['runoff_onset_anomaly'].where(ok) * w).sum(rm.BIN_DIMS) / w.sum(rm.BIN_DIMS).where(w.sum(rm.BIN_DIMS) > 0)
open_anom = (open_ds['runoff_onset_anomaly'].where(ok) * w).sum(rm.BIN_DIMS) / w.sum(rm.BIN_DIMS).where(w.sum(rm.BIN_DIMS) > 0)
enough_bins = ok.sum(rm.BIN_DIMS) >= MIN_MATCHED_BINS
forest_anom_df = forest_anom.where(enough_bins).to_pandas()
open_anom_df = open_anom.where(enough_bins).to_pandas()
temperature_df = statistics_df[[f'spring_temperature_anomaly_WY{y}' for y in WATER_YEARS]].set_axis(WATER_YEARS, axis=1)
rows = {}
for r in forest_anom_df.index:
    d = pd.DataFrame({'f': forest_anom_df.loc[r].values, 'o': open_anom_df.loc[r].values,
                      't': temperature_df.loc[r].values if r in temperature_df.index else np.nan}).dropna()
    if len(d) < MIN_MATCHED_YEARS:
        continue
    rows[r] = {'matched_years': len(d), 'anomaly_r_forest_open': np.corrcoef(d['f'], d['o'])[0, 1],
               'anomaly_sd_forest': d['f'].std(), 'anomaly_sd_open': d['o'].std(),
               'temperature_slope_forest': sps.linregress(d['t'], d['f']).slope,
               'temperature_slope_open': sps.linregress(d['t'], d['o']).slope}
by_range_df = by_range_df.join(pd.DataFrame.from_dict(rows, orient='index'))
by_range_df.loc[compared, list(rows[next(iter(rows))])].describe().round(2)

## 4. Summary, by latitude band

In [ ]:
c = by_range_df[compared].copy()
c['abs_latitude_band'] = rm.by_latitude_band(c.index.map(statistics_df['centroid_latitude']).to_series(index=c.index).abs()).astype(str)
both_lapse = c.dropna(subset=['lapse_rate_open', 'lapse_rate_forest'])
both_amp = c.dropna(subset=['sunny_shady_open_d', 'sunny_shady_forest_d'])
inter = c.dropna(subset=['anomaly_r_forest_open'])
summary = {
    'n_ranges_compared': int(len(c)), 'median_forest_share': round(float(c['forest_share'].median()), 3),
    'onset_forest_minus_open_median_d': round(float(c['onset_forest_minus_open_d'].median()), 1),
    'onset_forest_minus_open_iqr_d': [round(float(v), 1) for v in c['onset_forest_minus_open_d'].quantile([.25, .75])],
    'onset_forest_minus_open_min_d': round(float(c['onset_forest_minus_open_d'].min()), 1),
    'share_forest_earlier': round(float((c['onset_forest_minus_open_d'] < 0).mean()), 2),
    'mad_forest_minus_open_median_d': round(float(c['mad_forest_minus_open_d'].median()), 1),
    'lapse_rate_median_open_forest': [round(float(both_lapse['lapse_rate_open'].median()), 2), round(float(both_lapse['lapse_rate_forest'].median()), 2)],
    'lapse_rate_wilcoxon_p': float(sps.wilcoxon(both_lapse['lapse_rate_forest'], both_lapse['lapse_rate_open']).pvalue),
    'sunny_shady_median_open_forest_d': [round(float(both_amp['sunny_shady_open_d'].median()), 1), round(float(both_amp['sunny_shady_forest_d'].median()), 1)],
    'sunny_shady_wilcoxon_p': float(sps.wilcoxon(both_amp['sunny_shady_forest_d'], both_amp['sunny_shady_open_d']).pvalue),
    'n_ranges_interannual': int(len(inter)),
    'anomaly_r_forest_open_median': round(float(inter['anomaly_r_forest_open'].median()), 2),
    'temperature_slope_median_forest_open': [round(float(inter['temperature_slope_forest'].median()), 1), round(float(inter['temperature_slope_open'].median()), 1)],
    'by_abs_latitude_band': c.groupby('abs_latitude_band')['onset_forest_minus_open_d'].agg(['size', 'median']).round(1).to_dict(orient='index'),
    '_provenance': results.provenance(config)}
with open(results_dir / 'forest_vs_open_summary.json', 'w') as f:
    json.dump(summary, f, indent=1)
by_range_df.reset_index().assign(**results.provenance(config)).to_csv(results_dir / 'forest_vs_open_by_range.csv', index=False, float_format='%.4g')
summary